# Transformer from Scratch — Attention Is All You Need

Build the **core building block** behind GPT, BERT, and all modern NLP/CV models.

- **Self-Attention** mechanism step-by-step
- **Multi-Head Attention**
- **Positional Encoding**
- **Transformer Encoder Block**
- **Text classification** with a Transformer

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')
print(f'TensorFlow: {tf.__version__}')

## 1. Self-Attention — The Core Idea

For each word, compute how much to **attend** to every other word.

```
Query (Q) = "What am I looking for?"
Key (K)   = "What do I contain?"
Value (V) = "What information do I provide?"

Attention(Q, K, V) = softmax(Q · Kᵀ / √d_k) · V
```

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Core attention mechanism."""
    d_k = tf.cast(tf.shape(K)[-1], tf.float32)
    
    # Step 1: Q · Kᵀ (similarity scores)
    scores = tf.matmul(Q, K, transpose_b=True)  # (batch, seq_q, seq_k)
    
    # Step 2: Scale by √d_k (prevent large values)
    scores = scores / tf.math.sqrt(d_k)
    
    # Step 3: Mask (optional, for decoder)
    if mask is not None:
        scores += (mask * -1e9)
    
    # Step 4: Softmax → attention weights
    attention_weights = tf.nn.softmax(scores, axis=-1)
    
    # Step 5: Weighted sum of values
    output = tf.matmul(attention_weights, V)
    
    return output, attention_weights

# Demo with a simple example
np.random.seed(42)
seq_len, d_model = 4, 8
Q = tf.random.normal((1, seq_len, d_model))
K = tf.random.normal((1, seq_len, d_model))
V = tf.random.normal((1, seq_len, d_model))

output, weights = scaled_dot_product_attention(Q, K, V)
print(f'Input shape:    {Q.shape}')
print(f'Output shape:   {output.shape}')
print(f'Weights shape:  {weights.shape}')

# Visualize attention weights
plt.figure(figsize=(5, 4))
plt.imshow(weights[0].numpy(), cmap='Blues', vmin=0, vmax=1)
plt.colorbar(label='Attention Weight')
plt.xlabel('Key Position'); plt.ylabel('Query Position')
plt.title('Self-Attention Weights')
plt.xticks(range(seq_len)); plt.yticks(range(seq_len))
plt.tight_layout()
plt.show()

## 2. Multi-Head Attention

Instead of one attention function, use **multiple heads** that attend to different representation subspaces.

```
MultiHead(Q, K, V) = Concat(head₁, head₂, ..., headₕ) · Wᴼ
where headᵢ = Attention(Q·Wᵢᵠ, K·Wᵢᴷ, V·Wᵢⱽ)
```

In [ ]:
class MultiHeadAttention(layers.Layer):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % num_heads == 0
        self.depth = d_model // num_heads
        
        self.wq = layers.Dense(d_model)
        self.wk = layers.Dense(d_model)
        self.wv = layers.Dense(d_model)
        self.dense = layers.Dense(d_model)
    
    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])  # (batch, heads, seq, depth)
    
    def call(self, v, k, q, mask=None):
        batch_size = tf.shape(q)[0]
        
        q = self.split_heads(self.wq(q), batch_size)
        k = self.split_heads(self.wk(k), batch_size)
        v = self.split_heads(self.wv(v), batch_size)
        
        attention_output, weights = scaled_dot_product_attention(q, k, v, mask)
        
        attention_output = tf.transpose(attention_output, perm=[0, 2, 1, 3])
        concat = tf.reshape(attention_output, (batch_size, -1, self.d_model))
        
        return self.dense(concat), weights

# Test
mha = MultiHeadAttention(d_model=64, num_heads=8)
x = tf.random.normal((2, 10, 64))  # (batch=2, seq=10, d_model=64)
out, _ = mha(x, x, x)
print(f'Multi-Head Attention output: {out.shape}')

## 3. Positional Encoding

Transformers have no notion of order → **positional encoding** adds position info.

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [ ]:
def positional_encoding(max_len, d_model):
    positions = np.arange(max_len)[:, np.newaxis]
    dims = np.arange(d_model)[np.newaxis, :]
    angles = positions / np.power(10000, (2 * (dims // 2)) / d_model)
    
    pe = np.zeros((max_len, d_model))
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return pe

pe = positional_encoding(100, 64)
plt.figure(figsize=(12, 4))
plt.imshow(pe, cmap='RdBu', aspect='auto')
plt.colorbar(label='Value')
plt.xlabel('Embedding Dimension'); plt.ylabel('Position')
plt.title('Positional Encoding (100 positions × 64 dims)')
plt.tight_layout()
plt.show()

## 4. Transformer Encoder Block

In [ ]:
class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attention = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(d_model)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)
    
    def call(self, inputs, training=False):
        # Multi-Head Attention + Residual + LayerNorm
        attn_output = self.attention(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        
        # Feed-Forward + Residual + LayerNorm
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)


class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)
    
    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        return self.token_emb(x) + self.pos_emb(positions)

## 5. Text Classification with Transformer

In [ ]:
# Load IMDB dataset
vocab_size = 20000
maxlen = 200

(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(num_words=vocab_size)
X_train = keras.preprocessing.sequence.pad_sequences(X_train, maxlen=maxlen)
X_test = keras.preprocessing.sequence.pad_sequences(X_test, maxlen=maxlen)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# Build Transformer classifier
embed_dim = 64
num_heads = 4
ff_dim = 128

inputs = layers.Input(shape=(maxlen,))
embedding = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)(inputs)
transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)(embedding)
x = layers.GlobalAveragePooling1D()(transformer_block)
x = layers.Dropout(0.1)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

transformer_model = keras.Model(inputs=inputs, outputs=outputs)
transformer_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
transformer_model.summary()

In [ ]:
# Train
history = transformer_model.fit(
    X_train, y_train, epochs=5, batch_size=64,
    validation_data=(X_test, y_test), verbose=1
)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Transformer — IMDB Sentiment Classification', fontsize=14)
plt.tight_layout()
plt.show()

test_loss, test_acc = transformer_model.evaluate(X_test, y_test, verbose=0)
print(f'\nTest Accuracy: {test_acc:.4f}')

## 6. Transformer Architecture Summary

```
┌──────────────────────────────────────┐
│          Transformer Encoder         │
├──────────────────────────────────────┤
│  Input Embedding + Positional Enc    │
│           ↓                          │
│  ┌─ Multi-Head Self-Attention ────┐  │
│  │  Q, K, V → Scaled Dot-Product  │  │
│  └────────────────────────────────┘  │
│  + Residual Connection + LayerNorm   │
│           ↓                          │
│  ┌─ Feed-Forward Network ─────────┐  │
│  │  Dense → ReLU → Dense          │  │
│  └────────────────────────────────┘  │
│  + Residual Connection + LayerNorm   │
│           ↓                          │
│  (Repeat × N layers)                 │
└──────────────────────────────────────┘
```

### Key Models Built on Transformers:
- **BERT**: Encoder-only (bidirectional), used for classification, NER, QA
- **GPT**: Decoder-only (left-to-right), used for text generation
- **T5**: Encoder-Decoder, used for any text-to-text task
- **ViT**: Vision Transformer, applies attention to image patches